# NBA Data Cleaning Pipeline

**Goal:** Clean and normalize 4 raw NBA datasets for loading into PostgreSQL (star schema) and preparation for feature engineering / ML.

**Cleaning order:**
```
1. Games.csv                     ← anchor table (fact_games)
2. TeamStatistics.csv            ← fact_team_game_stats
3. PlayerStatisticsAdvanced.csv  ← fact_player_game_advanced
4. PlayerStatisticsScoring.csv   ← fact_player_game_scoring
```

This order guarantees referential integrity: the parent (`Games`) is cleaned before its children.

> Raw files in `Data_raw/` are read-only. All outputs go to `Data_processed/` as Parquet.

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

RAW   = '../Data_raw/'
CLEAN = '../Data_processed/'

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print('Setup complete.')

pandas 2.2.3 | numpy 2.1.3
Setup complete.


---
## 1. Games.csv

**Role in the model:** `fact_games` — central anchor table. Each row is a unique game.

| Keep | Drop |
|---|---|
| game_id, game_datetime, home/away/winner team_id, scores, game_type, attendance | city/name cols (denormalized), arena cols, officials (free text), always-empty cols: gameSubtype, gameLabel, gameSubLabel, seriesGameNumber |

In [2]:
# 1.1 Load — low_memory=False prevents DtypeWarning from mixed-type columns
games_raw = pd.read_csv(RAW + 'Games.csv', low_memory=False)
print(f'Shape: {games_raw.shape}')

# 1.2 Initial exploration
print('--- dtypes ---')
print(games_raw.dtypes.to_string())

print('--- Nulls per column ---')
null_counts = games_raw.isnull().sum()
print(null_counts[null_counts > 0])

print('--- Always-empty columns (empty string) ---')
empty_counts = (games_raw == '').sum()
print(empty_counts[empty_counts > 0])

print(f'Duplicates: {games_raw.duplicated().sum()}')
games_raw.head(3)

Shape: (72864, 22)
--- dtypes ---
gameId                int64
gameDateTimeEst      object
hometeamCity         object
hometeamName         object
hometeamId            int64
awayteamCity         object
awayteamName         object
awayteamId            int64
homeScore             int64
awayScore             int64
winner                int64
gameType             object
gameSubtype          object
gameLabel            object
gameSubLabel         object
seriesGameNumber    float64
attendance          float64
arenaId               int64
arenaName            object
arenaCity            object
arenaState           object
officials            object
--- Nulls per column ---


hometeamCity            7
awayteamCity            7
gameSubtype         72790
gameLabel           68912
gameSubLabel        72627
seriesGameNumber    67126
attendance          71885
arenaName           71879
arenaCity           71879
arenaState          71888
officials           71879
dtype: int64
--- Always-empty columns (empty string) ---
Series([], dtype: int64)
Duplicates: 0


,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,...,gameSubtype,gameLabel,gameSubLabel,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials
0,22500878,2026-03-01 21:30:00,Los Angeles,Lakers,1610612747,Sacramento,Kings,1610612758,128,104,...,NaN,NaN,NaN,NaN,18272.0,1000069,Crypto.com Arena,Los Angeles,CA,"Mitchell Ervin, Andy Nagy, Pat O'Connell"
1,22500877,2026-03-01 21:00:00,LA,Clippers,1610612746,New Orleans,Pelicans,1610612740,137,117,...,NaN,NaN,NaN,NaN,17003.0,1000137,Intuit Dome,Inglewood,CA,"Ed Malloy, Dedric Taylor, Mousa Dagher"
2,22500876,2026-03-01 20:00:00,Dallas,Mavericks,1610612742,Oklahoma City,Thunder,1610612760,87,100,...,NaN,NaN,NaN,NaN,18900.0,193,American Airlines Center,Dallas,TX,"Scott Twardoski, Tyler Ford, Phenizee Ransom"


In [3]:
# 1.3 Column selection and renaming
GAMES_COLS = {
    'gameId':          'game_id',
    'gameDateTimeEst': 'game_datetime',
    'hometeamId':      'home_team_id',
    'awayteamId':      'away_team_id',
    'homeScore':       'home_score',
    'awayScore':       'away_score',
    'winner':          'winner_team_id',
    'gameType':        'game_type',
    'attendance':      'attendance',
}

games = games_raw[list(GAMES_COLS.keys())].rename(columns=GAMES_COLS).copy()
print(f'Columns selected : {len(games.columns)}')
print(f'Columns dropped  : {games_raw.shape[1] - len(games.columns)}')

# 1.4 Type transformations
games['game_datetime']  = pd.to_datetime(games['game_datetime'])
games['game_id']        = games['game_id'].astype('int64')
games['home_team_id']   = games['home_team_id'].astype('int64')
games['away_team_id']   = games['away_team_id'].astype('int64')
games['winner_team_id'] = games['winner_team_id'].astype('int64')
games['home_score']     = games['home_score'].astype('int16')
games['away_score']     = games['away_score'].astype('int16')
games['game_type']      = games['game_type'].astype('category')

# attendance can have nulls — Int32 (capital) is pandas nullable integer
games['attendance'] = pd.to_numeric(games['attendance'], errors='coerce').astype('Int32')

# Remove unplayed games (score 0-0): cancelled or postponed
zero_score_mask = (games['home_score'] == 0) & (games['away_score'] == 0)
n_cancelled = zero_score_mask.sum()
if n_cancelled > 0:
    print(f'Removing {n_cancelled} games with score 0-0 (cancelled/postponed):')
    print(games[zero_score_mask][['game_id', 'game_datetime', 'game_type']].to_string())
    games = games[~zero_score_mask].copy()

# 1.5 Derived columns
# NBA season spans two calendar years (Oct-Jun)
# Oct/Nov/Dec 2025 → 2025-26 → season_end_year = 2026
# Jan-Jun 2026     → 2025-26 → season_end_year = 2026
games['season_end_year']   = games['game_datetime'].apply(
    lambda x: x.year + 1 if x.month >= 10 else x.year
).astype('int16')

games['margin_of_victory'] = (games['home_score'] - games['away_score']).astype('int16')
games['total_points']      = (games['home_score'] + games['away_score']).astype('int16')
games['home_win']          = (games['winner_team_id'] == games['home_team_id']).astype('int8')

# Note: is_overtime is added in Section 2 (requires quarter data from TeamStatistics)

# 1.6 Scope filter: last 10 seasons (2017-2026)
# Reason: modern advanced stats, elimination of historical NaN, ML relevance
SEASON_CUTOFF = 2017
before_filter = len(games)
games = games[games['season_end_year'] >= SEASON_CUTOFF].copy()
print(f'Time filter: {before_filter:,} → {len(games):,} games (seasons >= {SEASON_CUTOFF})')
print(f'Seasons kept: {sorted(games["season_end_year"].unique())}')
print(f'Global home win rate: {games["home_win"].mean():.3f}')
games.dtypes

Columns selected : 9
Columns dropped  : 13
Removing 2 games with score 0-0 (cancelled/postponed):
       game_id       game_datetime       game_type
1174  22400988 2025-04-01 20:00:00  Regular Season
1271  22400627 2025-03-19 21:00:00  Regular Season


Time filter: 72,862 → 13,151 games (seasons >= 2017)
Seasons kept: [np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024), np.int16(2025), np.int16(2026)]
Global home win rate: 0.561


game_id                       int64
game_datetime        datetime64[ns]
home_team_id                  int64
away_team_id                  int64
home_score                    int16
away_score                    int16
winner_team_id                int64
game_type                  category
attendance                    Int32
season_end_year               int16
margin_of_victory             int16
total_points                  int16
home_win                       int8
dtype: object

In [4]:
# 1.6 Integrity validations
assert games['game_id'].notna().all(),       'ERROR: game_id has nulls'
assert games['game_id'].is_unique,           'ERROR: game_id is not unique (PK violated)'
assert games['home_team_id'].notna().all(),  'ERROR: home_team_id has nulls'
assert games['away_team_id'].notna().all(),  'ERROR: away_team_id has nulls'
assert games['winner_team_id'].notna().all(),'ERROR: winner_team_id has nulls'

all_team_ids = pd.concat([games['home_team_id'], games['away_team_id']])
assert games['winner_team_id'].isin(all_team_ids).all(), \
    'ERROR: winner_team_id is neither home_team_id nor away_team_id'

# Data anomalies: games with margin=0 (ties) that should not exist in NBA
# Not removed — documented for traceability. Likely data provider errors.
ties = games[games['margin_of_victory'] == 0]
if len(ties) > 0:
    print(f'ANOMALY documented: {len(ties)} games with tied score (data error):')
    print(ties[['game_id', 'game_datetime', 'home_score', 'away_score', 'winner_team_id']].to_string())

print(f'\nOK games clean   : {len(games):,} rows | {len(games.columns)} columns')
print(f'   Seasons        : {games["season_end_year"].min()} - {games["season_end_year"].max()}')
print(f'   game_type      : {dict(games["game_type"].value_counts())}')
print(f'   attendance null: {games["attendance"].isna().sum()} games')

# 1.7 Export — Parquet: compressed, typed, 10x faster than CSV on read
games.to_parquet(CLEAN + 'games_clean.parquet', index=False)
print('Saved: Data_processed/games_clean.parquet')
games.head(3)


OK games clean   : 13,151 rows | 13 columns
   Seasons        : 2017 - 2026
   game_type      : {'Regular Season': np.int64(11583), 'Playoffs': np.int64(748), 'Preseason': np.int64(714), 'NBA Emirates Cup': np.int64(66), 'Play-in Tournament': np.int64(31), 'All-Star Game': np.int64(7), 'NBA Cup': np.int64(1), 'Emirates NBA Cup': np.int64(1)}
   attendance null: 12172 games
Saved: Data_processed/games_clean.parquet


,game_id,game_datetime,home_team_id,away_team_id,home_score,away_score,winner_team_id,game_type,attendance,season_end_year,margin_of_victory,total_points,home_win
0,22500878,2026-03-01 21:30:00,1610612747,1610612758,128,104,1610612747,Regular Season,18272,2026,24,232,1
1,22500877,2026-03-01 21:00:00,1610612746,1610612740,137,117,1610612746,Regular Season,17003,2026,20,254,1
2,22500876,2026-03-01 20:00:00,1610612742,1610612760,87,100,1610612760,Regular Season,18900,2026,-13,187,0


---
## 2. TeamStatistics.csv

**Role in the model:** `fact_team_game_stats` — 2 rows per game (home + away). Composite PK: `(game_id, team_id)`.

**Critical note:** the `turnovers` column contains empty strings `''` (not NaN) in some records. Requires explicit conversion with `pd.to_numeric(..., errors='coerce')`.

| Keep | Drop |
|---|---|
| game_id, team_id, opponent_team_id, is_home, is_win, scores, box score stats, quarter points, bench_pts, biggest_lead, fast break pts, pts from TOs, pts in paint, season record | gameDateTimeEst, city/name cols, numMinutes (always 240), coachId (always empty), biggestScoringRun, leadChanges, timesTied, timeoutsRemaining, pointsSecondChance |

In [5]:
# 2.1 Load
ts_raw = pd.read_csv(RAW + 'TeamStatistics.csv', low_memory=False)
print(f'Shape: {ts_raw.shape}  ->  2 rows per game (home + away)')

# 2.2 Exploration
print('--- Nulls per column ---')
null_counts_ts = ts_raw.isnull().sum()
print(null_counts_ts[null_counts_ts > 0])

print('--- Columns with empty strings ---')
empty_ts = (ts_raw == '').sum()
print(empty_ts[empty_ts > 0])

print(f'Duplicates (gameId, teamId): {ts_raw.duplicated(["gameId", "teamId"]).sum()}')

game_counts = ts_raw.groupby('gameId').size()
print(f'Rows per game (distribution): {dict(game_counts.value_counts())}')

ts_raw.head(4)

Shape: (145730, 48)  ->  2 rows per game (home + away)
--- Nulls per column ---
teamCity                       16
teamName                        2
opponentTeamCity               16
opponentTeamName                2
win                             2
assists                     35089
blocks                      40703
steals                      41147
fieldGoalsAttempted         34308
fieldGoalsMade                 20
fieldGoalsPercentage        34389
threePointersAttempted      40790
threePointersMade           29858
threePointersPercentage     41497
freeThrowsAttempted          5098
freeThrowsMade                 22
freeThrowsPercentage         5104
reboundsDefensive           41479
reboundsOffensive           41332
reboundsTotal               34746
foulsPersonal                3994
turnovers                   40817
plusMinusPoints              1014
numMinutes                     10
q1Points                   141010
q2Points                   141022
q3Points                   141026
q4

,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,...,leadChanges,pointsFastBreak,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId
0,22500878,2026-03-01 21:30:00,Los Angeles,Lakers,1610612747,Sacramento,Kings,1610612758,1,1.0,...,1.0,20.0,27.0,50.0,9.0,1.0,1.0,36.0,24.0,NaN
1,22500878,2026-03-01 21:30:00,Sacramento,Kings,1610612758,Los Angeles,Lakers,1610612747,0,0.0,...,1.0,16.0,17.0,58.0,8.0,1.0,0.0,14.0,48.0,NaN
2,22500877,2026-03-01 21:00:00,New Orleans,Pelicans,1610612740,LA,Clippers,1610612746,0,0.0,...,0.0,13.0,9.0,50.0,16.0,0.0,0.0,19.0,43.0,NaN
3,22500877,2026-03-01 21:00:00,LA,Clippers,1610612746,New Orleans,Pelicans,1610612740,1,1.0,...,0.0,22.0,36.0,60.0,8.0,0.0,0.0,28.0,31.0,NaN


In [6]:
# 2.3 Column selection and renaming
TEAM_COLS = {
    'gameId':                  'game_id',
    'teamId':                  'team_id',
    'opponentTeamId':          'opponent_team_id',
    'home':                    'is_home',
    'win':                     'is_win',
    'teamScore':               'team_score',
    'opponentScore':           'opponent_score',
    'fieldGoalsMade':          'fg_made',
    'fieldGoalsAttempted':     'fg_attempted',
    'fieldGoalsPercentage':    'fg_pct',
    'threePointersMade':       'fg3_made',
    'threePointersAttempted':  'fg3_attempted',
    'threePointersPercentage': 'fg3_pct',
    'freeThrowsMade':          'ft_made',
    'freeThrowsAttempted':     'ft_attempted',
    'freeThrowsPercentage':    'ft_pct',
    'reboundsOffensive':       'reb_off',
    'reboundsDefensive':       'reb_def',
    'reboundsTotal':           'reb_total',
    'assists':                 'assists',
    'turnovers':               'turnovers',
    'steals':                  'steals',
    'blocks':                  'blocks',
    'foulsPersonal':           'fouls',
    'plusMinusPoints':         'plus_minus',
    'q1Points':                'q1_pts',
    'q2Points':                'q2_pts',
    'q3Points':                'q3_pts',
    'q4Points':                'q4_pts',
    'benchPoints':             'bench_pts',
    'biggestLead':             'biggest_lead',
    'pointsFastBreak':         'pts_fast_break',
    'pointsFromTurnovers':     'pts_from_turnovers',
    'pointsInThePaint':        'pts_in_paint',
    'seasonWins':              'season_wins',
    'seasonLosses':            'season_losses',
}

ts = ts_raw[list(TEAM_COLS.keys())].rename(columns=TEAM_COLS).copy()
print(f'Columns selected : {len(ts.columns)}')
print(f'Columns dropped  : {ts_raw.shape[1] - len(ts.columns)}')

# 2.4 Type transformations
# ARCHITECTURAL NOTE: many columns have NaN in historical records (pre-1970s)
# where advanced stats were not tracked. We use nullable types (capital letter)
# to preserve missing values without forcing 0 or dropping rows.
ts['game_id']          = ts['game_id'].astype('int64')
ts['team_id']          = ts['team_id'].astype('int64')
ts['opponent_team_id'] = ts['opponent_team_id'].astype('int64')

# Int8 (capital) for binary flags — is_win has 2 NaN in historical data
ts['is_home'] = pd.to_numeric(ts['is_home'], errors='coerce').astype(pd.Int8Dtype())
ts['is_win']  = pd.to_numeric(ts['is_win'],  errors='coerce').astype(pd.Int8Dtype())

# pd.to_numeric + Int16 (capital): handles NaN, empty strings and floats correctly
all_int16 = [
    'team_score', 'opponent_score', 'fg_made', 'fg_attempted',
    'fg3_made', 'fg3_attempted', 'ft_made', 'ft_attempted',
    'reb_off', 'reb_def', 'reb_total', 'assists', 'steals', 'blocks',
    'fouls', 'plus_minus', 'q1_pts', 'q2_pts', 'q3_pts', 'q4_pts',
    'bench_pts', 'biggest_lead', 'pts_fast_break', 'pts_from_turnovers',
    'pts_in_paint', 'season_wins', 'season_losses', 'turnovers'
]
for col in all_int16:
    ts[col] = pd.to_numeric(ts[col], errors='coerce').astype(pd.Int16Dtype())

for col in ['fg_pct', 'fg3_pct', 'ft_pct']:
    ts[col] = pd.to_numeric(ts[col], errors='coerce').astype('float32')

# 2.5 Derived columns
ts['score_margin']  = (ts['team_score'] - ts['opponent_score'])
ts['fg2_made']      = (ts['fg_made']     - ts['fg3_made'])
ts['fg2_attempted'] = (ts['fg_attempted'] - ts['fg3_attempted'])

# eFG% = (FGM + 0.5 * 3PM) / FGA — weights 3-pointers appropriately vs FG%
ts['efg_pct'] = (
    (ts['fg_made'].astype('float64') + 0.5 * ts['fg3_made'].astype('float64'))
    / ts['fg_attempted'].astype('float64')
).astype('float32')

# is_overtime: 1 if overtime occurred, 0 if not, NA if no quarter data available
# Logic: if team_score > sum of q1+q2+q3+q4 → game went to OT
q_total = ts[['q1_pts','q2_pts','q3_pts','q4_pts']].astype('float64').sum(axis=1, skipna=False)
has_q_data = q_total.notna()
ot_raw = (ts['team_score'].astype('float64') > q_total)
ts['is_overtime'] = ot_raw.where(has_q_data).astype(pd.Int8Dtype())  # NA where no quarter data

ot_games = int(ts.groupby('game_id')['is_overtime'].max().dropna().sum())
null_q = (~has_q_data).sum() // 2
print(f'Games without quarter data (historical): {null_q:,}')
print(f'Overtime games detected: {ot_games:,}')
ts.dtypes

Columns selected : 36
Columns dropped  : 12


Games without quarter data (historical): 70,513
Overtime games detected: 118


game_id                 int64
team_id                 int64
opponent_team_id        int64
is_home                  Int8
is_win                   Int8
team_score              Int16
opponent_score          Int16
fg_made                 Int16
fg_attempted            Int16
fg_pct                float32
fg3_made                Int16
fg3_attempted           Int16
fg3_pct               float32
ft_made                 Int16
ft_attempted            Int16
ft_pct                float32
reb_off                 Int16
reb_def                 Int16
reb_total               Int16
assists                 Int16
turnovers               Int16
steals                  Int16
blocks                  Int16
fouls                   Int16
plus_minus              Int16
q1_pts                  Int16
q2_pts                  Int16
q3_pts                  Int16
q4_pts                  Int16
bench_pts               Int16
biggest_lead            Int16
pts_fast_break          Int16
pts_from_turnovers      Int16
pts_in_pai

In [7]:
# 2.6 Validations

# STEP 1: Filter team_stats records with no valid game in games_clean
# This includes game_ids not present in Games.csv (cancelled, invalid)
games_ref = pd.read_parquet(CLEAN + 'games_clean.parquet')
before = len(ts)
ts = ts[ts['game_id'].isin(games_ref['game_id'])].copy()
n_removed = before - len(ts)
if n_removed > 0:
    print(f'Removed {n_removed} team_stats records with no match in games_clean')

# STEP 2: Verify PK and structure AFTER filtering
assert not ts.duplicated(['game_id', 'team_id']).any(), \
    'ERROR: composite PK (game_id, team_id) has duplicates'

game_counts_check = ts.groupby('game_id').size()
if not (game_counts_check == 2).all():
    bad = game_counts_check[game_counts_check != 2]
    print(f'ANOMALY: {len(bad)} games with != 2 teams (documenting, not removing):')
    print(bad)
else:
    print('OK: exactly 2 teams per game')

assert set(ts['game_id']).issubset(set(games_ref['game_id'])), \
    'ERROR: FK violated after filtering'

print(f'\nOK team_stats clean  : {len(ts):,} rows | {len(ts.columns)} columns')
print(f'   Unique games      : {ts["game_id"].nunique():,} (in games_clean: {len(games_ref):,})')

# 2.7 Propagate is_overtime back to games_clean
# is_overtime is derived from quarter data in team_stats but belongs logically to games
ot_flags = ts.groupby('game_id')['is_overtime'].max().reset_index()
games_with_ot = games_ref.merge(ot_flags, on='game_id', how='left')
games_with_ot['is_overtime'] = games_with_ot['is_overtime'].fillna(0).astype(pd.Int8Dtype())
games_with_ot.to_parquet(CLEAN + 'games_clean.parquet', index=False)

ot_count = int(games_with_ot['is_overtime'].dropna().sum())
print(f'games_clean updated with is_overtime')
print(f'   OT: {ot_count:,} of {len(games_with_ot):,} games ({100*ot_count/len(games_with_ot):.1f}%)')

# 2.8 Export
ts.to_parquet(CLEAN + 'team_stats_clean.parquet', index=False)
print('Saved: Data_processed/team_stats_clean.parquet')
ts.head(3)

Removed 119428 team_stats records with no match in games_clean
OK: exactly 2 teams per game

OK team_stats clean  : 26,302 rows | 41 columns
   Unique games      : 13,151 (in games_clean: 13,151)
games_clean updated with is_overtime
   OT: 118 of 13,151 games (0.9%)
Saved: Data_processed/team_stats_clean.parquet


,game_id,team_id,opponent_team_id,is_home,is_win,team_score,opponent_score,fg_made,fg_attempted,fg_pct,...,pts_fast_break,pts_from_turnovers,pts_in_paint,season_wins,season_losses,score_margin,fg2_made,fg2_attempted,efg_pct,is_overtime
0,22500878,1610612747,1610612758,1,1,128,104,47,94,0.500,...,20,27,50,36,24,24,29,55,0.595745,0
1,22500878,1610612758,1610612747,0,0,104,128,42,90,0.467,...,16,17,58,14,48,-24,36,64,0.500000,0
2,22500877,1610612740,1610612746,0,0,117,137,40,89,0.449,...,13,9,50,19,43,-20,28,56,0.516854,0


---
## 3. PlayerStatisticsAdvanced.csv

**Role in the model:** `fact_player_game_advanced` — advanced metrics per player/game. Composite PK: `(game_id, person_id)`.

**Note on source file:** the CSV covers data from 1997 onwards (unlike Games.csv which goes back to 1946). Advanced per-player stats began being tracked digitally around that era.

**Scope decision:** after the FK filter on `games_clean` (which already contains only 2017-2026), player stat tables are automatically scoped to the same time range. No additional season filter is needed.

| Keep | Drop |
|---|---|
| game_id, person_id, team_id, is_home, is_win, minutes, fg stats, efg_pct, ts_pct, usg_pct, ast_pct, ast_to_ratio, reb/oreb/dreb_pct, off/def/net_rating, pace, possessions, pie, team_tov_pct | firstName/lastName (→ dim_players), city/name cols, wl, minSec, teamName (empty), availableFlag (empty), fgaPg/fgmPg (redundant), estimated metrics (eOff/eDef/eNet, etc.), pacePer40, astRatio, sp_work_* (8 external ranking cols) |

In [8]:
# 3.1 Load
adv_raw = pd.read_csv(RAW + 'PlayerStatisticsAdvanced.csv', low_memory=False)
print(f'Shape: {adv_raw.shape}')

# gameDateTimeEst has mixed types — convert before calling .min()/.max()
_dt = pd.to_datetime(adv_raw['gameDateTimeEst'], errors='coerce')
print(f'Time range: {_dt.min()} -> {_dt.max()}')

# 3.2 Exploration
print('--- Nulls per column (with nulls only) ---')
null_adv = adv_raw.isnull().sum().sort_values(ascending=False)
print(null_adv[null_adv > 0].head(15))

print(f'Duplicates (gameId, personId): {adv_raw.duplicated(["gameId", "personId"]).sum()}')

print('--- Key metrics statistics ---')
adv_raw[['efgPct', 'tsPct', 'offRating', 'defRating', 'netRating', 'usgPct']].describe().T

Shape: (116898, 60)
Time range: 1997-04-24 19:00:00 -> 2026-02-06 22:00:00
--- Nulls per column (with nulls only) ---
teamName            116898
availableFlag        56092
gameDateTimeEst       3435
playerteamName        3435
opponentteamName      3435
win                   3435
opponentteamCity      3435
home                  3435
playerName             813
nickname               813
lastName               581
firstName              577
playerteamCity         436
matchup                140
wl                      49
dtype: int64
Duplicates (gameId, personId): 0
--- Key metrics statistics ---


,count,mean,std,min,25%,50%,75%,max
efgPct,116898.0,0.456533,0.298484,0.0,0.267,0.500,0.636,1.5
tsPct,116898.0,0.497026,0.285869,0.0,0.333,0.509,0.667,1.5
offRating,116898.0,101.162857,26.122514,0.0,88.900,102.600,116.000,400.0
defRating,116898.0,103.367568,25.918704,0.0,90.200,103.500,117.000,400.0
netRating,116898.0,-2.204563,36.118799,-400.0,-19.200,0.000,16.700,400.0
usgPct,116872.0,0.183653,0.088906,0.0,0.127,0.181,0.237,1.0


In [9]:
# 3.3 Column selection and renaming
ADV_COLS = {
    'gameId':    'game_id',
    'personId':  'person_id',
    'teamId':    'team_id',
    'home':      'is_home',
    'win':       'is_win',
    'min':       'minutes',
    'fgm':       'fg_made',
    'fga':       'fg_attempted',
    'fgPct':     'fg_pct',
    'efgPct':    'efg_pct',
    'tsPct':     'ts_pct',
    'usgPct':    'usg_pct',
    'astPct':    'ast_pct',
    'astTo':     'ast_to_ratio',
    'orebPct':   'oreb_pct',
    'drebPct':   'dreb_pct',
    'rebPct':    'reb_pct',
    'offRating': 'off_rating',
    'defRating': 'def_rating',
    'netRating': 'net_rating',
    'pace':      'pace',
    'poss':      'possessions',
    'pie':       'pie',
    'tmTovPct':  'team_tov_pct',
}

adv = adv_raw[list(ADV_COLS.keys())].rename(columns=ADV_COLS).copy()
print(f'Columns selected : {len(adv.columns)}')
print(f'Columns dropped  : {adv_raw.shape[1] - len(adv.columns)}')

# 3.4 Type transformations
adv['game_id']   = adv['game_id'].astype('int64')
adv['person_id'] = adv['person_id'].astype('int64')
adv['team_id']   = adv['team_id'].astype('int64')

# home and win have 3,435 NaN in historical data — use nullable Int8
adv['is_home'] = pd.to_numeric(adv['is_home'], errors='coerce').astype(pd.Int8Dtype())
adv['is_win']  = pd.to_numeric(adv['is_win'],  errors='coerce').astype(pd.Int8Dtype())

adv['minutes']      = pd.to_numeric(adv['minutes'],      errors='coerce').astype('float32')
adv['fg_made']      = pd.to_numeric(adv['fg_made'],      errors='coerce').astype(pd.Int16Dtype())
adv['fg_attempted'] = pd.to_numeric(adv['fg_attempted'], errors='coerce').astype(pd.Int16Dtype())
adv['possessions']  = pd.to_numeric(adv['possessions'],  errors='coerce').astype(pd.Int16Dtype())

float32_cols = [
    'fg_pct', 'efg_pct', 'ts_pct', 'usg_pct', 'ast_pct', 'ast_to_ratio',
    'oreb_pct', 'dreb_pct', 'reb_pct', 'off_rating', 'def_rating', 'net_rating',
    'pace', 'pie', 'team_tov_pct'
]
for col in float32_cols:
    adv[col] = pd.to_numeric(adv[col], errors='coerce').astype('float32')

# 3.5 Statistical outlier treatment
# efg_pct and ts_pct are normally in [0, 1.5]. Values > 1.5 are mathematically impossible
# and indicate source errors. Replace with NaN — do not impute or clip.
n_efg_out = (adv['efg_pct'] > 1.5).sum()
n_ts_out  = (adv['ts_pct']  > 1.5).sum()
print(f'Outliers in efg_pct (> 1.5): {n_efg_out}')
print(f'Outliers in ts_pct  (> 1.5): {n_ts_out}')

adv['efg_pct'] = adv['efg_pct'].where(adv['efg_pct'] <= 1.5)
adv['ts_pct']  = adv['ts_pct'].where(adv['ts_pct']  <= 1.5)
print('Outliers set to NaN (not imputed — error is in the source)')

# 3.6 Derived columns — season_end_year from game_id
# NBA game_id format: {1-digit type}{2-digit season_start_year}{5-digit game_num}
# Scope is 2017-2026: year_2d is always 17-26, so century prefix is always 2000.
# e.g.: 22500878 -> str[1:3]='25' -> 2000+25+1 = 2026
def season_end_year_from_game_id(gid):
    return 2000 + int(str(int(gid))[1:3]) + 1

adv['season_end_year'] = adv['game_id'].apply(season_end_year_from_game_id).astype('int16')

print('Most recent seasons:')
print(adv['season_end_year'].value_counts().sort_index(ascending=False).head(10))

Columns selected : 24
Columns dropped  : 36
Outliers in efg_pct (> 1.5): 0
Outliers in ts_pct  (> 1.5): 0
Outliers set to NaN (not imputed — error is in the source)


Most recent seasons:
season_end_year
2100     1475
2099     1339
2098     1424
2097     1412
2026    18823
2025     4033
2024     3883
2023     3820
2022     3816
2021     3337
Name: count, dtype: int64


In [10]:
# 3.7 FK filter: keep only games present in games_clean (scope 2017-2026)
games_ref = pd.read_parquet(CLEAN + 'games_clean.parquet')
before = len(adv)
adv = adv[adv['game_id'].isin(games_ref['game_id'])].copy()
print(f'FK filter (games_clean): {before:,} → {len(adv):,} records')

# 3.8 Validations
assert not adv.duplicated(['game_id', 'person_id']).any(), \
    'ERROR: composite PK (game_id, person_id) has duplicates'
assert adv['game_id'].notna().all(),   'ERROR: game_id has nulls'
assert adv['person_id'].notna().all(), 'ERROR: person_id has nulls'

# FK must be 100% after filter
orphan_adv = set(adv['game_id']) - set(games_ref['game_id'])
assert len(orphan_adv) == 0, f'ERROR: {len(orphan_adv)} orphan game_ids after filter'

print(f'OK player_adv clean   : {len(adv):,} rows | {len(adv.columns)} columns')
print(f'   Seasons            : {adv["season_end_year"].min()} - {adv["season_end_year"].max()}')
print(f'   Unique players     : {adv["person_id"].nunique():,}')

# 3.9 Export
adv.to_parquet(CLEAN + 'player_stats_advanced_clean.parquet', index=False)
print('Saved: Data_processed/player_stats_advanced_clean.parquet')
adv.head(3)

FK filter (games_clean): 116,898 → 53,113 records
OK player_adv clean   : 53,113 rows | 25 columns
   Seasons            : 2017 - 2026
   Unique players     : 1,709
Saved: Data_processed/player_stats_advanced_clean.parquet


,game_id,person_id,team_id,is_home,is_win,minutes,fg_made,fg_attempted,fg_pct,efg_pct,...,dreb_pct,reb_pct,off_rating,def_rating,net_rating,pace,possessions,pie,team_tov_pct,season_end_year
62206,41600405,2747,1610612739,0,0,40.799999,9,11,0.818,1.136,...,0.083,0.040,132.100006,132.899994,-0.800000,99.370003,84,0.132,0.000000,2017
62207,41600405,202684,1610612739,0,0,29.900000,6,8,0.750,0.750,...,0.138,0.143,128.300003,135.500000,-7.200000,98.040001,60,0.134,18.799999,2017
62208,41600405,201142,1610612744,1,1,40.299999,14,20,0.700,0.825,...,0.125,0.078,135.800003,115.000000,20.799999,96.000000,81,0.215,12.500000,2017


---
## 4. PlayerStatisticsScoring.csv

**Role in the model:** `fact_player_game_scoring` — scoring breakdown by shot type per player/game. Composite PK: `(game_id, person_id)`. Complements `PlayerStatisticsAdvanced`.

| Keep | Drop |
|---|---|
| game_id, person_id, team_id, is_home, is_win, minutes, fg stats, pct_fga_2pt/3pt, pct_pts_2pt/3pt/ft/paint/fastbreak/off_tov, pct_ast_fgm, pct_unast_fgm | firstName/lastName, city/name cols, wl, minSec, teamName (empty), availableFlag, gameDate/gameDateTimeEst/gameType, matchup, pctAst2Pm/3Pm (sub-granular), pctUast2Pm/3Pm, pctPts2PtMr (ambiguous column) |

In [11]:
# 4.1 Load
scr_raw = pd.read_csv(RAW + 'PlayerStatisticsScoring.csv', low_memory=False)
print(f'Shape: {scr_raw.shape}')

# 4.2 Exploration
print('--- Nulls per column ---')
null_scr = scr_raw.isnull().sum()
print(null_scr[null_scr > 0])

print(f'Duplicates (gameId, personId): {scr_raw.duplicated(["gameId", "personId"]).sum()}')

print('--- Percentage columns statistics ---')
pct_cols_preview = [c for c in scr_raw.columns if c.startswith('pct')]
scr_raw[pct_cols_preview].describe().T

Shape: (116898, 42)
--- Nulls per column ---
firstName              577
lastName               581
playerteamCity         436
playerteamName        3435
availableFlag        56092
gameDateTimeEst       3435
home                  3435
matchup                140
nickname               813
opponentteamCity      3435
opponentteamName      3435
playerName             813
teamName            116898
win                   3435
wl                      49
dtype: int64
Duplicates (gameId, personId): 0
--- Percentage columns statistics ---


,count,mean,std,min,25%,50%,75%,max
pctAst2Pm,116898.0,0.405934,0.400476,0.000,0.000,0.333,0.750,1.0
pctAst3Pm,116898.0,0.352830,0.463265,0.000,0.000,0.000,1.000,1.0
pctAstFgm,116898.0,0.520079,0.389311,0.000,0.000,0.500,1.000,1.0
pctFga2Pt,116898.0,0.639446,0.331914,0.000,0.429,0.667,1.000,1.0
pctFga3Pt,116898.0,0.298336,0.298229,0.000,0.000,0.250,0.500,1.0
pctPts2Pt,116898.0,0.486126,0.358567,0.000,0.125,0.500,0.800,1.0
pctPts2PtMr,116898.0,0.120758,0.215159,0.000,0.000,0.000,0.182,1.0
pctPts3Pt,116898.0,0.209550,0.298787,0.000,0.000,0.000,0.400,1.0
pctPtsFb,116898.0,0.108506,0.196640,0.000,0.000,0.000,0.167,1.0
pctPtsFt,116898.0,0.304324,0.353156,-0.001,0.000,0.182,0.428,1.0


In [12]:
# 4.3 Column selection and renaming
SCR_COLS = {
    'gameId':       'game_id',
    'personId':     'person_id',
    'teamId':       'team_id',
    'home':         'is_home',
    'win':          'is_win',
    'min':          'minutes',
    'fgm':          'fg_made',
    'fga':          'fg_attempted',
    'fgPct':        'fg_pct',
    'pctFga2Pt':    'pct_fga_2pt',
    'pctFga3Pt':    'pct_fga_3pt',
    'pctPts2Pt':    'pct_pts_2pt',
    'pctPts3Pt':    'pct_pts_3pt',
    'pctPtsFt':     'pct_pts_ft',
    'pctPtsPaint':  'pct_pts_paint',
    'pctPtsFb':     'pct_pts_fastbreak',
    'pctPtsOffTov': 'pct_pts_off_tov',
    'pctAstFgm':    'pct_ast_fgm',
    'pctUastFgm':   'pct_unast_fgm',
}

scr = scr_raw[list(SCR_COLS.keys())].rename(columns=SCR_COLS).copy()
print(f'Columns selected : {len(scr.columns)}')
print(f'Columns dropped  : {scr_raw.shape[1] - len(scr.columns)}')

# 4.4 Type transformations
scr['game_id']   = scr['game_id'].astype('int64')
scr['person_id'] = scr['person_id'].astype('int64')
scr['team_id']   = scr['team_id'].astype('int64')

# home and win have NaN in historical data — use nullable Int8
scr['is_home'] = pd.to_numeric(scr['is_home'], errors='coerce').astype(pd.Int8Dtype())
scr['is_win']  = pd.to_numeric(scr['is_win'],  errors='coerce').astype(pd.Int8Dtype())

scr['minutes']      = pd.to_numeric(scr['minutes'],      errors='coerce').astype('float32')
scr['fg_made']      = pd.to_numeric(scr['fg_made'],      errors='coerce').astype(pd.Int16Dtype())
scr['fg_attempted'] = pd.to_numeric(scr['fg_attempted'], errors='coerce').astype(pd.Int16Dtype())
scr['fg_pct']       = pd.to_numeric(scr['fg_pct'],       errors='coerce').astype('float32')

pct_cols = [
    'pct_fga_2pt', 'pct_fga_3pt', 'pct_pts_2pt', 'pct_pts_3pt', 'pct_pts_ft',
    'pct_pts_paint', 'pct_pts_fastbreak', 'pct_pts_off_tov',
    'pct_ast_fgm', 'pct_unast_fgm'
]
for col in pct_cols:
    scr[col] = pd.to_numeric(scr[col], errors='coerce').astype('float32')

# 4.5 Derived columns
# season_end_year reuses the function defined in Section 3
scr['season_end_year'] = scr['game_id'].apply(season_end_year_from_game_id).astype('int16')

print('Seasons in player_scoring:')
print(scr['season_end_year'].value_counts().sort_index())
scr.dtypes

Columns selected : 19
Columns dropped  : 23


Seasons in player_scoring:
season_end_year
2001     1394
2002     1369
2003     1765
2004     2023
2005     2801
2006     3971
2007     3996
2008     4399
2009     4514
2010     4551
2011     4464
2012     2497
2013     4691
2014     4778
2015     4612
2016     4731
2017     4432
2018     3868
2019     3955
2020     4725
2021     3337
2022     3816
2023     3820
2024     3883
2025     4033
2026    18823
2097     1412
2098     1424
2099     1339
2100     1475
Name: count, dtype: int64


game_id                int64
person_id              int64
team_id                int64
is_home                 Int8
is_win                  Int8
minutes              float32
fg_made                Int16
fg_attempted           Int16
fg_pct               float32
pct_fga_2pt          float32
pct_fga_3pt          float32
pct_pts_2pt          float32
pct_pts_3pt          float32
pct_pts_ft           float32
pct_pts_paint        float32
pct_pts_fastbreak    float32
pct_pts_off_tov      float32
pct_ast_fgm          float32
pct_unast_fgm        float32
season_end_year        int16
dtype: object

In [13]:
# 4.6 FK filter: keep only games present in games_clean (scope 2017-2026)
games_ref = pd.read_parquet(CLEAN + 'games_clean.parquet')
before = len(scr)
scr = scr[scr['game_id'].isin(games_ref['game_id'])].copy()
print(f'FK filter (games_clean): {before:,} → {len(scr):,} records')

# 4.7 Validations
assert not scr.duplicated(['game_id', 'person_id']).any(), \
    'ERROR: composite PK (game_id, person_id) has duplicates'
assert scr['game_id'].notna().all(),   'ERROR: game_id has nulls'
assert scr['person_id'].notna().all(), 'ERROR: person_id has nulls'

orphan_scr = set(scr['game_id']) - set(games_ref['game_id'])
assert len(orphan_scr) == 0, f'ERROR: {len(orphan_scr)} orphan game_ids after filter'

# Verify symmetry with player_adv — both tables should cover the exact same games/players
adv_ref   = pd.read_parquet(CLEAN + 'player_stats_advanced_clean.parquet')
pairs_scr = set(zip(scr['game_id'], scr['person_id']))
pairs_adv = set(zip(adv_ref['game_id'], adv_ref['person_id']))
print(f'(game_id, person_id) pairs only in scoring : {len(pairs_scr - pairs_adv)}')
print(f'(game_id, person_id) pairs only in advanced: {len(pairs_adv - pairs_scr)}')

print(f'OK player_scoring clean  : {len(scr):,} rows | {len(scr.columns)} columns')
print(f'   Unique players        : {scr["person_id"].nunique():,}')

# 4.8 Export
scr.to_parquet(CLEAN + 'player_stats_scoring_clean.parquet', index=False)
print('Saved: Data_processed/player_stats_scoring_clean.parquet')
scr.head(3)

FK filter (games_clean): 116,898 → 53,113 records
(game_id, person_id) pairs only in scoring : 0
(game_id, person_id) pairs only in advanced: 0
OK player_scoring clean  : 53,113 rows | 20 columns
   Unique players        : 1,709
Saved: Data_processed/player_stats_scoring_clean.parquet


,game_id,person_id,team_id,is_home,is_win,minutes,fg_made,fg_attempted,fg_pct,pct_fga_2pt,pct_fga_3pt,pct_pts_2pt,pct_pts_3pt,pct_pts_ft,pct_pts_paint,pct_pts_fastbreak,pct_pts_off_tov,pct_ast_fgm,pct_unast_fgm,season_end_year
0,22500745,1629057,1610612757,1,1,19.100000,6,6,1.0,0.833,0.167,0.769,0.231,0.0,0.769,0.154,0.0,0.667,0.333,2026
1,22500742,1629750,1610612765,1,1,13.500000,3,3,1.0,0.333,0.667,0.250,0.750,0.0,0.250,0.250,0.0,1.000,0.000,2026
2,22500742,1629011,1610612752,0,0,20.200001,2,2,1.0,1.000,0.000,1.000,0.000,0.0,1.000,0.000,0.0,1.000,0.000,2026


---
## 5. Final Cross-Validation

Loads all 4 clean files and verifies referential integrity, unique PKs, row counts, and memory usage.

In [14]:
# Load all 4 clean files
games_final = pd.read_parquet(CLEAN + 'games_clean.parquet')
ts_final    = pd.read_parquet(CLEAN + 'team_stats_clean.parquet')
adv_final   = pd.read_parquet(CLEAN + 'player_stats_advanced_clean.parquet')
scr_final   = pd.read_parquet(CLEAN + 'player_stats_scoring_clean.parquet')

print('=' * 60)
print('CLEAN FILES SUMMARY')
print('=' * 60)
datasets = [
    ('games_clean',           games_final),
    ('team_stats_clean',      ts_final),
    ('player_stats_advanced', adv_final),
    ('player_stats_scoring',  scr_final),
]
for name, df in datasets:
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f'  {name:<28}: {len(df):>10,} rows | {len(df.columns):>3} cols | {mb:>6.1f} MB')

print()
print('--- FK Checks ---')
ok_ts = set(ts_final['game_id']).issubset(set(games_final['game_id']))
print(f'  team_stats  -> games : {"OK" if ok_ts else "FAIL"}')

pct_adv = len(set(adv_final['game_id']) & set(games_final['game_id'])) / adv_final['game_id'].nunique() * 100
pct_scr = len(set(scr_final['game_id']) & set(games_final['game_id'])) / scr_final['game_id'].nunique() * 100
print(f'  player_adv  -> games : {pct_adv:.1f}% covered')
print(f'  player_scr  -> games : {pct_scr:.1f}% covered')

print()
print('--- PK Checks ---')
pk_checks = [
    ('games',          games_final, ['game_id']),
    ('team_stats',     ts_final,    ['game_id', 'team_id']),
    ('player_adv',     adv_final,   ['game_id', 'person_id']),
    ('player_scoring', scr_final,   ['game_id', 'person_id']),
]
for name, df, pk in pk_checks:
    dupes = df.duplicated(pk).sum()
    status = 'OK' if dupes == 0 else f'FAIL ({dupes} duplicates)'
    print(f'  {name:<18} PK {str(pk):<35}: {status}')

print()
print('--- team_stats structure ---')
rows_per_game = ts_final.groupby('game_id').size()
ok_structure = (rows_per_game == 2).all()
print(f'  Exactly 2 teams per game: {"OK" if ok_structure else "FAIL"}')

print()
print('PIPELINE COMPLETE')
print('Files ready in Data_processed/ for loading into PostgreSQL.')

CLEAN FILES SUMMARY
  games_clean                 :     13,151 rows |  14 cols |    0.8 MB
  team_stats_clean            :     26,302 rows |  41 cols |    3.7 MB
  player_stats_advanced       :     53,113 rows |  25 cols |    5.5 MB
  player_stats_scoring        :     53,113 rows |  20 cols |    4.5 MB

--- FK Checks ---
  team_stats  -> games : OK
  player_adv  -> games : 100.0% covered
  player_scr  -> games : 100.0% covered

--- PK Checks ---
  games              PK ['game_id']                        : OK
  team_stats         PK ['game_id', 'team_id']             : OK
  player_adv         PK ['game_id', 'person_id']           : OK
  player_scoring     PK ['game_id', 'person_id']           : OK

--- team_stats structure ---
  Exactly 2 teams per game: OK

PIPELINE COMPLETE
Files ready in Data_processed/ for loading into PostgreSQL.
